In [ ]:
import os
#-----Section 1: Core Imports------#
#These are basic imports required to read the 3572 base pairs blueprint#
from Bio import SeqIO #Reads the .fasta or .gb files
from Bio.Seq import Seq #Handles the genetic Strings#
from Bio.SeqRecord import SeqRecord #Maintains Metadata During Analysis#


#----Section 2: Structural Audit Imports-------#
#These are crucial to ensure the engineered protein keeps its pl of 5.78 and 0.0087#
from Bio.SeqUtils import ProtParam            # Core protein analysis module
from Bio.SeqUtils.ProtParam import ProteinAnalysis # Class for MW (Molecular Weight), pI (Isoelectronic Point), and GRAVY (Hydrophobicity Score)
from Bio.SeqUtils import gc_fraction #This calculates the fraction of guanine's and cytosines in a DNA / RNA Sequences#

#------Section 3: Molecular Engineering Imports------#
#These ensure it can be manufactured. These scan for restriction sites that could "break" during lab assembly#
from Bio.Restriction import BseRI, BpmI, PacI, AgeI, RestrictionBatch #This imports all the restriction enzymes#
from Bio.Restriction import Analysis #This allows to run a full "digest" simulation#

#------Section 4: Safety and Expression Imports-----#
from Bio.SeqUtils import CodonAdaptationIndex # To calculate Codon Adaptation Index (CAI)
from Bio.Blast import NCBIWWW # To run the remote safety BLAST against the human genome
from Bio.Blast import NCBIXML # To parse and report safety results

#------Section 5: Machine Learning imports--------#
import torch 
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim #This will aid in importing the Adam Optimizer#
import random #Will aid in generating the synthetic data#
import numpy as np
from Bio.PDB import PDBParser
from Bio import Entrez
from torch.utils.data import Dataset, DataLoader, random_split
import glob
from dataclasses import dataclass
import math
# i will import the followiing in order to avoid useless error messages
import warnings

In [2]:
#Think of this as a "Quality Control" test. This the basic first step before creating the PLM (Protein Language Model)#
#What to test for (Just Testing the metrics)#
#1. Genetic Fidelity & Frame Check [Passed!]
#2. Physical Property Verification [Passed!]
#3. Manufacturing & Restriction Audit [Passed!]
#4. Transcriptome-wide Safety Scan [Passed!]

In [2]:
#Genetic Fidelity and Frame Check#
#The program below slices teh palsmid and translates it#

#Step 1: Load the Fasta file#
file_name = "vhl-crispr-cas13-system.fasta"
record = next(SeqIO.parse(file_name, "fasta")) #This parses the fasta file and makes it into SeqRecords to maintaimn the data#
plasmid_seq = record.seq

#Step 2: Extracting the sequence from bp 3030 to the end

vhl_dna = plasmid_seq[3029:] #This goes from bp 3029 to end#

#Step 3: Translating the DNA to protein#

vhl_protein = vhl_dna.translate(to_stop = False) #to stop indicates if there is any stop codons before the end#

#Check 1 - Main Check: The Audit#
print(f"Extracted DNA length: {len(vhl_dna)} bp")
print(f"Translated Protein Length: {len(vhl_protein)} residues")
#The results prove well because every codon sequence is 3 base pairs and 543 / 3 = 181 residues which is accurate#


#Subcheck 1: Start Sequence (must be MPRK)
start_match  = vhl_protein.startswith("MPRK")
print(f"\nStarts with 'MPRK': {'Yes! it starts with MPRK!' if start_match else 'No! It starts with (' + str(vhl_protein[:4]) + ')'}")

#Subcheck 2: Internal Stop Codons
#The following checks for stop codons in the sequence itself but ignores if it comes at teh end of the codon sequence#
stops = vhl_protein.count("*") 
premature_stop = "*" in vhl_protein[:-1]
print(f"\nPremature Stop Codons: {'YES' if premature_stop else 'NONE'}")

#Subcheck 3: Exact residue count
length_correct = (len(vhl_protein.rstrip("*")) == 181)
print(f"\nLength is 181 residues: {'YES' if length_correct else 'NONE'}")

if start_match and not premature_stop and length_correct:
    print("\nRESULT: Phase 1 Passed. Genetic Fidelity Confirmed.")
else:
    print("\nRESULT: Phase 1 Failed. Check your frame or sequence coordinates.")

Extracted DNA length: 543 bp
Translated Protein Length: 181 residues

Starts with 'MPRK': Yes! it starts with MPRK!

Premature Stop Codons: NONE

Length is 181 residues: YES

RESULT: Phase 1 Passed. Genetic Fidelity Confirmed.


In [4]:
#Physical Properties Verification#
#This sees if the physical properties are proper for the VHL cargo to work#

polished_protein = str(vhl_protein).strip("*") #Thsi removes any training of the stop codon so it does not interfere with the caluclations#

vhl_analysis = ProteinAnalysis(polished_protein) #This creates the ProtPartam analysis and decleares the variable#

#--------Caluclations----------------#
# Molecular Weight
mw_da = vhl_analysis.molecular_weight()
mw_kda = mw_da / 1000

# Isoelectric Point (pI)
isoelectronic_point = vhl_analysis.isoelectric_point()

# Hydrophobicity (GRAVY Score)
# GRAVY = Grand Average of Hydropathy
gravy_score = vhl_analysis.gravy()

print(f"--- BIOPHYSICAL SIGNATURE CHECK ---")
print(f"Molecular Weight: {mw_da:.2f} Da ({mw_kda:.2f} kDa)")
print(f"Isoelectric Point (pI): {isoelectronic_point:.2f}")
print(f"GRAVY Score: {gravy_score:.4f}") #There is a difference of +0.0087 compared to teh original VHL sequence#

--- BIOPHYSICAL SIGNATURE CHECK ---
Molecular Weight: 20630.13 Da (20.63 kDa)
Isoelectric Point (pI): 5.78
GRAVY Score: -0.6287


In [5]:
#Manufacturing and Restriction Auditing#
#This is basically a cloning safety check to ensure the protein is perfect without any accidental restriction sites#

vhl_dna = plasmid_seq[3029:] #Reusing code to ensure proper results

audit_enzymes = RestrictionBatch([BseRI, BpmI, PacI, AgeI]) #These are a all the "forbidden enzymes" that must nto be present in the cargo / protein zone#

results = audit_enzymes.search(vhl_dna) #Searches for the cutting sites in the enzymes listed in teh variabel audit_enzymes#

#-----------Restriction Auditing Results-----------#

forbidden_found = False #If the variable turns true it indicates that there is a cutting enzyme and need to change# 

for enzyme,sites in results.items():
    if len(sites) > 0:
        forbidden_found = True
        print(f"CRITICAL: {enzyme} found at position(s) {sites} inside VHL!")
    else:
        print(f"✅ CLEAR: No {enzyme} sites found.")

if not forbidden_found:
    print("\nRESULT: VHL cargo is 'Clean' and ready for manufacturing.")
else:
    print("\nRESULT: Action Required! You must perform synonymous mutations.")

    #The BTSL can be ignored because it is foudn naturally foudn in the original DNA sequence#


✅ CLEAR: No BseRI sites found.
✅ CLEAR: No BpmI sites found.
✅ CLEAR: No PacI sites found.
✅ CLEAR: No AgeI sites found.

RESULT: VHL cargo is 'Clean' and ready for manufacturing.


In [6]:
#Transciriptome-Wide Safety Scan#
#This checks if the spacer sequence might accidnetaly bind to healthy RNA transcripts

spacer_sequence = "cgtcaccctggatgtgtcctgcctcaaggg" #This is the 30 Nucleotide Spacer#

print("Going through the NCBI BLAST servers....")

result_handle = NCBIWWW.qblast(
    program="blastn", 
    database="refseq_rna", 
    sequence= spacer_sequence,
    entrez_query="human[orgn]", # Entrez_query filters for humans to avoid matches in other species#
    short_query=True 
)

blast_record = NCBIXML.read(result_handle) #This will parse the results on teh NCBI Database#
e_value_threshold = 0.05 #Thisi is the standard threshold becaus etehre should be no hits#

significant_hits = 0

for alignment in blast_record.alignments:
    for hsp in alignment.hsps:
        if hsp.expect < e_value_threshold:
            if "von Hippel-Lindau" not in alignment.title: #It is curucial to expect a hit on our tarhget anythign else is a red flag#
                print("Potential off-target found!")
                print(f"Target: {alignment.title}")
                print(f"E-value: {hsp.expect}")
                significant_hits += 1

if significant_hits == 0:
    print("\nSafety Scan Complete: No significant off-target human RNA matches found.")

#Results show that my 30 nucleotide spacer is unique to Von-Hippel Lindau gene itself.#
#The result indicate that my 230 nucleotide spacer does nto share homology with any other human genome like hemoglobin idnicating it will nto bind#

Going through the NCBI BLAST servers....


c:\Users\Admin\.conda\envs\pytorch_env\lib\site-packages\Bio\Blast\NCBIWWW.py:151: BiopythonWarning: "SHORT_QUERY_ADJUST" is incorrectly implemented (by NCBI) for blastn. We bypass the problem by manually adjusting the search parameters. Thus, results may slightly differ from web page searches.
  warnings.warn(



Safety Scan Complete: No significant off-target human RNA matches found.


In [10]:
#------------------------Machine Learning Model: Simulating "VHL Specific" CAS-13 in a human-like environment------------#
#I will run CNN models and many other types of machine learnign model to prove this really does work#

In [7]:
#Test #1 - I will test the 30 nucleotide spacer in a "human" environemnt#

#The first step includes convertuing the manually imported 30-nt spacer sequence and vconvertign it into numerical tensors#

ml_spacer_sequence = "cgtcaccctggatgtgtcctgcctcaaggg" #Reimporting to ensure independence of sequence when running not relying on the previous#
mapping = {'A': 0, 'C': 1, 'G': 2, 'U': 3, 'T': 3} #The following creates mapping numbers in order to create a 4 x 30 tensor with each nucleotide representing one vector. Moreover, Uracil and Thymien have same numerical value since both are similar but uracil is used for RNA#

#Creation of tensor input#

spacer_tensor = torch.zeros(len(ml_spacer_sequence), 4)
for i, base in enumerate(ml_spacer_sequence.upper()):
    spacer_tensor[i, mapping[base]] = 1.0

#i will now reshape the tensor to abide with the convolutional neural network (CNN) input#
#It makes teh tesnor coordinates into [Batch, Channels, Length]
spacer_CNN = spacer_tensor.t().unsqueeze(0) #This is purely for CNN Inpout

In [8]:
#-----------Simulation using Gibbs Free Energy / Analyzing thermodynamic potential----------#
#For this cell, I will test the 3572 bp Cas13. I WILL ATTACH MY MATHEMATICAL PROOF AS WELL AS A PNG FILE#
#All my calaculations will be proven for each step of code#
Cas13fasta = "vhl-crispr-cas13-system.fasta"
record = next(SeqIO.parse(Cas13fasta, "fasta"))
full_sequence = str(record.seq).upper() #Records all the stirngs as upper case letters in order to avoid errors when calculating base energy .Some base pairs are lowercase letters in the FASTA sequence.#

def thermodynamic_potential(sequence, window_size = 30, temp = 310): #The temeprature has been set to 310 Kelvin#
    R = 0.001987 #R repsresnts the ideal gas constant measured in kilocalories / mol * Kelvin# 

    #I will now map the nucleotides based on Guanine and Cytosine content and convert the Fasta sequences into pytorch tensors#
    #the below is known as nearest Neighbor Stacking Energies.Tehse are simplified biological values for#
    NN_Energy = {
        'AA': -0.9, 'AU': -0.9, 'AC': -2.1, 'AG': -1.7,
        'UU': -0.9, 'UA': -1.3, 'UC': -2.1, 'UG': -2.1,
        'CC': -3.3, 'CA': -2.1, 'CU': -2.1, 'CG': -3.4,
        'GG': -3.3, 'GA': -2.1, 'GU': -2.1, 'GC': -3.4,
        'TT': -0.9, 'TA': -1.3, 'TC': -2.1, 'TG': -2.1,
        'AT': -0.9, 'CT': -2.1, 'GT': -2.1, 'UT': -0.9
    }

    # 1. Map sequence to PAIR energies
    #Here I am basically calculating teh ent
    pair_energies = [NN_Energy.get(sequence[i:i+2], -1.0) for i in range(len(sequence)-1)]
    energy_tensor = torch.tensor(pair_energies, dtype=torch.float32)

    # 2. Vectorized Windowing 
    # Use window_size - 1 to get the correct number of stacking pairs
    unfolded_tensor = energy_tensor.unfold(0, window_size - 1, 1)

    # 3. Sum and Scale
    raw_window_sum = unfolded_tensor.sum(dim=1)
    effective_binding = raw_window_sum * 0.45

    initiation_penalty = 4.1  # Cost of the first base pair
    electrostatic_penalty = 0.15 * window_size

    adjusted_dg = effective_binding + initiation_penalty + electrostatic_penalty

    A = 1e5# The frequency of molecular vibrations at 310K
    k_off = A * torch.exp(adjusted_dg / (R * temp))

    return adjusted_dg, k_off #Returns the values clauclated in the function#

dg_scores, off_rates = thermodynamic_potential(full_sequence)

print(f"Validated Analysis of {len(full_sequence)} bp system.")
print(f"Physiologically Accurate ΔG: {torch.min(dg_scores):.2f} kcal/mol")
print(f"Realistic Off-rate: {torch.mean(off_rates):.2e} s^-1")

#I will now find the specific index / bp where the 30 bp sequence will bidn into rna
best_index = torch.argmin(dg_scores).item()

optimized_sequence = full_sequence[best_index : best_index + 30]

print(f"\nLocation in Cas13 which has the best bidning potential: Start Position {best_index}") #After 896 it goes on for the next 30 base pairs# 

Validated Analysis of 3572 bp system.
Physiologically Accurate ΔG: -32.89 kcal/mol
Realistic Off-rate: 1.59e-03 s^-1

Location in Cas13 which has the best bidning potential: Start Position 896


In [9]:
#-----------------CNN (Convolutional Neural Network) Building--------------#
#I will genereate synthetic data of G-depletion and Seed Regions and then train the CNN to prodcue results with an adam optimizer to ensure high-qualioty results#
#My model will run thre
torch.manual_seed(42)

#Below i have 
def encode_sequence(seq):
    mapping = {'A': 0, 'U': 1, 'C': 2, 'G': 3, 'T': 1}
    tensor = torch.zeros(4, 30)
    for i, base in enumerate(seq[:30]):
        if base in mapping:
            tensor[mapping[base], i] = 1.0
    return tensor.unsqueeze(0) # Returns shape [1, 4, 30]

# Synthetic Data Generation ---
def synthetic_data(num_samples=100):
    data = [] 
    labels_list = []
    
    for _ in range(num_samples):
        efficiency = random.choice([0, 1])
        if efficiency:
            #  G-depletion (<20% G) is GOOD for Cas13
            seq = ''.join(random.choices(['A', 'U', 'C'], k=25) + random.choices(['G'], k=5))
            target_val = 1.0
        else:
            # High G-content is BAD (causes RNA knots)
            seq = ''.join(random.choices(['G'], k=20) + random.choices(['A', 'U', 'C'], k=10))
            target_val = 0.0
        
        data.append(encode_sequence(seq))
        labels_list.append(torch.tensor([[target_val]]))
    
    return data, labels_list

class Cas13Predictor(nn.Module):
    def __init__(self):
        super(Cas13Predictor, self).__init__()
        # Scans 4 channels (A,U,C,G) with 16 filters using a 3-base window
        self.conv = nn.Conv1d(4, 16, kernel_size=3)
        # Flattened input: 16 filters * (30nt - 2 from conv) = 448
        self.fc = nn.Linear(16 * 28, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = torch.relu(self.conv(x))
        x = x.view(x.size(0), -1) 
        return self.sigmoid(self.fc(x))

model = Cas13Predictor()

train_data, train_labels = synthetic_data(200)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCELoss()

print("Training CNN on Biological Rules (G-Depletion & Seed Sensitivity)...")
for epoch in range(21):
    total_loss = 0
    for i in range(len(train_data)):
        optimizer.zero_grad()
        output = model(train_data[i])
        loss = criterion(output, train_labels[i])
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if epoch % 5 == 0:
        print(f"Epoch {epoch} | Loss: {total_loss/200:.4f}")

# Final Prediction VHL Sequence
efficiency_score = model(spacer_CNN)
print(f"\nTargeted VHL Efficiency (Post-Training): {efficiency_score.item():.4f}")

Training CNN on Biological Rules (G-Depletion & Seed Sensitivity)...
Epoch 0 | Loss: 0.2691
Epoch 5 | Loss: 0.0003
Epoch 10 | Loss: 0.0001
Epoch 15 | Loss: 0.0000
Epoch 20 | Loss: 0.0000

Targeted VHL Efficiency (Post-Training): 0.6981


In [ ]:
#-------------Checking The Driving Expression of the EF-1-alpha promoter and Intron A--------------#
#The EF (Expression Factor) contains specific transcription factors. It is possible ot use a 1D-CNN with a kernel size of 8-12 to be a motif detector#
#I will test the splicign efficiency and the GC-Richness since EF-1-alpha is prone to methylation# 

In [10]:
torch.manual_seed(42)

# 1. THE ARCHITECTURE
class ExpressionInfrastructureModel(nn.Module):
    def __init__(self):
        super(ExpressionInfrastructureModel, self).__init__()

        # Infrastructure Branch (Promoter/Intron A)
        self.promoter_cnn = nn.Sequential(
            nn.Conv1d(4, 32, kernel_size=11, padding=5),
            nn.ReLU(),
            nn.MaxPool1d(4),
            nn.Conv1d(32, 64, kernel_size=7, padding=3),
            nn.AdaptiveAvgPool1d(1)
        )

        # Cargo Branch (Cas13/VHL)
        self.cargo_cnn = nn.Sequential(
            nn.Conv1d(4, 32, kernel_size=11, padding=5),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)
        )

        # Integration Layer
        self.fc = nn.Linear(64 + 32, 1)

    def forward(self, full_sequence):
        # Slicing the 3,572 bp sequence into functional heads
        promoter = full_sequence[:, :, :1000]
        cargo = full_sequence[:, :, 3000:] 

        promoter_feat = self.promoter_cnn(promoter).squeeze(-1)
        cargo_feat = self.cargo_cnn(cargo).squeeze(-1)

        combined = torch.cat((promoter_feat, cargo_feat), dim=1)
        expression_score = torch.sigmoid(self.fc(combined))
        return expression_score


def dna_to_onehot(sequence):
    mapping = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    seq_str = str(sequence).upper()
    one_hot = np.zeros((4, len(seq_str)), dtype=np.float32)
    for i, base in enumerate(seq_str):
        if base in mapping:
            one_hot[mapping[base], i] = 1.0
    return torch.tensor(one_hot).unsqueeze(0)


#Uses the variable plasmid_seq from a preciosu cell to create a tensor 
vhl_tensor = dna_to_onehot(plasmid_seq)

model = ExpressionInfrastructureModel()
optimizer = optim.Adam(model.parameters(), lr=0.005) #I use an adam optimizer to ensure high leve lreustls and decide teh learnign rate#
criterion = nn.BCELoss()

print("Calibration Results:")
model.train()
for epoch in range(50):
    optimizer.zero_grad()
    
    # Positive Pass
    pos_out = model(vhl_tensor)
    pos_loss = criterion(pos_out, torch.tensor([[1.0]]))
    
    # Negative Pass: Scrambled infrastructure should fail (0.0)
    # This proves the model is sensitive to the promoter arrangement
    neg_vhl = vhl_tensor.clone()
    scramble_idx = torch.randperm(600)
    neg_vhl[:, :, :600] = neg_vhl[:, :, :600][:, :, scramble_idx]
    
    neg_out = model(neg_vhl)
    neg_loss = criterion(neg_out, torch.tensor([[0.0]]))
    
    total_loss = pos_loss + neg_loss
    total_loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f"Calibration Epoch {epoch} | Loss: {total_loss.item():.4f}")

#Below prints teh final resutls of the model#
model.eval()
with torch.no_grad():
    final_score = model(vhl_tensor).item()

print(f"\nInfrastructure Score: {final_score:.4f}")

# Interpretation logic#
if final_score > 0.50:
    status = "FUNCTIONAL BASELINE - READY!"
    insight = "Infrastructure recognized my model as biologically viable."
else:
    status = "OPTIMIZATION REQUIRED"
    insight = "Model identifies potential transcriptional interference."

print(f"\n{status}")
print(f"Core Insight: {insight}")


Calibration Results:
Calibration Epoch 0 | Loss: 1.3943
Calibration Epoch 10 | Loss: 1.3792
Calibration Epoch 20 | Loss: 1.3597
Calibration Epoch 30 | Loss: 1.2290
Calibration Epoch 40 | Loss: 0.5348

Infrastructure Score: 0.9602

FUNCTIONAL BASELINE - READY!
Core Insight: Infrastructure recognized my model as biologically viable.


In [ ]:
#-------------Transformer-BAsed RNA Accesibility Encoder--------------#
#I will now chekc if the VHL mRNA is actually open for binding#
#I would liek to give credit to NCBI RefSeq and RNAsolo because I have gotten bulk of my data from there#

In [ ]:
#1. Parsing all the data#
#I have 5724 sets of data to train this model and most of them are in dbn format#
#This will be done across ew cells to ensure proper parsing of all data#
#Teh will be done within teh folders themselves because they are huge sets of data#
#The results prove to show I have a strong set of RNA data#
#Moreover, the use of PDB files is highly useful for trainign purposes3

In [9]:
@dataclass
class DBNRecord:
    name: str
    sequence: str
    structure: str

def parse_single_dbn(filepath):
    with open(filepath) as f:
        lines = [l.strip() for l in f if l.strip()]
    name, seq, struct = None, None, None
    for line in lines:
        if line.startswith('>'):
            name = line[1:].strip()
        elif line.startswith('seq '):
            seq = line[4:].strip().upper().replace('T','U')
        elif line.startswith('cWW '):
            struct = line[4:].strip()
    if not all([name, seq, struct]):
        return None
    if len(seq) != len(struct) or len(seq) == 0:
        return None
    return DBNRecord(name=name, sequence=seq, structure=struct)

FOLDERS = [
    'RF00001_15_1_dbn',
    'RF00005_15_1_dbn',
    'BGSU__R__All__A__3_0__dbn_4_31',
]

all_records = []
for folder in FOLDERS:
    files = sorted(glob.glob(os.path.join(folder, '*.dbn')))
    records = [r for f in files if (r := parse_single_dbn(f))]
    print(f"{folder}: {len(records)} records")
    all_records.extend(records)

print(f"\nTotal: {len(all_records)} records")

# Extract windows + labels
FLANK, TARGET_LEN = 10, 30
WINDOW = FLANK + TARGET_LEN + FLANK

sequences, labels = [], []
for r in all_records:
    if len(r.sequence) < WINDOW:
        continue
    access = [1.0 if c == '.' else 0.0 for c in r.structure]
    for i in range(FLANK, len(r.sequence) - TARGET_LEN - FLANK, 5):
        sequences.append(r.sequence[i-FLANK : i+TARGET_LEN+FLANK])
        labels.append(sum(access[i:i+TARGET_LEN]) / TARGET_LEN)

print(f"Windows: {len(sequences)}")
print(f"Mean accessibility: {sum(labels)/len(labels):.3f}")

RF00001_15_1_dbn: 2151 records
RF00005_15_1_dbn: 3149 records
BGSU__R__All__A__3_0__dbn_4_31: 237 records

Total: 5537 records
Windows: 45019
Mean accessibility: 0.491


In [15]:
#---------Building The Transformation Encoder----------3
#When building the the Encoder I will do them cell by cell because there are maany steps involved in the process#

In [10]:
# Step 1: Building the actual model#
#I will first build the architecture of the model using biderectional and attention-nbased moddeling#

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len = 512, dropout = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.pe = pe.unsqueeze(0)


    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])
    
#This will be the Main Accesibility Predictor model and I will be using 2 transformation layers and 8 attention heads along with an emobodying vector of 64
class RNAAccessibilityPredictor(nn.Module): 
    def __init__(self, input_dim = 4, d_model = 64, nhead = 4, num_layers = 2, d_ff = 256, dropout = 0.1):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_enc = PositionalEncoding(d_model, dropout = dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model = d_model, nhead = nhead, dim_feedforward = d_ff,
            dropout = dropout, batch_first = True, norm_first = True
        )

        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers = num_layers)

        self.head = nn.Sequential(
            nn.Linear(d_model, 32), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1), nn.Sigmoid()
        )

    def forward(self, x):
        x = self.input_proj(x)
        x = self.pos_enc(x)
        x = self.encoder(x)
        x = x.mean(dim=1)
        return self.head(x).squeeze(-1)

In [11]:
NUC_MAP  = {'A':0, 'U':1, 'G':2, 'C':3}

def encode(seq):
    arr = np.zeros((len(seq), 4), dtype = np.float32)
    for i, c in enumerate(seq):
        if c in NUC_MAP:
            arr [i, NUC_MAP[c]] = 1.0
        else:
            arr[i, :] = 0.25
    return arr

class RNADataset(Dataset):
    def __init__(self, seqs, labels):
        self.seqs = seqs
        self.labels = labels
    def __len__(self):
        return len(self.seqs)
    def __getitem__(self, idx):
        x = torch.tensor(encode(self.seqs[idx]))
        y = torch.tensor(self.labels[idx], dtype=torch.float32)
        return x, y
    
dataset = RNADataset(sequences, labels)
n = len(dataset)
n_train = int(0.8 * n)
n_val = int(0.1 * n)
n_test = n - n_train - n_val

In [ ]:
#----------Training Model--------------------------#
warnings.filterwarnings('ignore')

train_ds, val_ds, test_ds = random_split(dataset, [n_train, n_val, n_test])

train_dl = DataLoader(train_ds, batch_size=64, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=64)

model = RNAAccessibilityPredictor(dropout = 0.1)
optimizer = torch.optim.Adam(model.parameters(), lr = 3e-4, weight_decay= 1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max = 50)

#Take in mind the model has 45019 windows imn three groups 
print(f"Training (What the Model Actually Learns From): {n_train}") #80% of the window data goes towards teh trainign variable#
print(f"Val (Check if teh model was overfitted during the Training): {n_val}") #10% is checked to see if the model is overfitted#
print(f"Test (The Final Score Achieved after the training has been completed): {n_test}") #The last 10% goes towards pullign a ascore after teh trainign is done#
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}\n")


for epoch in range (1, 4):
    model.train()
    train_loss = 0

    for x, y in train_dl:
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss == loss.item()

    model.eval()
    val_loss = 0
    with torch.no_grad():
        for x, y in val_dl:
            val_loss += criterion(model(x), y).item()
    
    scheduler.step()

    if epoch == 3:
        print(f"Epoch {epoch:3d} | Train: {train_loss/len(train_dl):.4f} | Val: {val_loss/len(val_dl):.4f}")

Training (What the Model Actually Learns From): 36015
Val (Check if teh model was overfitted during the Training): 4501
Test (The Final Score Achieved after the training has been completed): 4503
Parameters: 102,401

Epoch   3 | Train: 0.0000 | Val: 0.6921

Saved → accessibility_model.pt


In [13]:
test_dl = DataLoader(test_ds, batch_size=64)
model.eval()
preds_all, labels_all = [], []
with torch.no_grad():
    for x, y in test_dl:
        preds_all.extend(model(x).tolist())
        labels_all.extend(y.tolist())

mae = sum(abs(p-l) for p,l in zip(preds_all, labels_all)) / len(preds_all)
print(f"Test MAE: {mae:.4f}")
print("< 0.10 = excellent | < 0.15 = good | > 0.20 = needs more data")

Test MAE: 0.0830
< 0.10 = excellent | < 0.15 = good | > 0.20 = needs more data


In [14]:
record     = next(SeqIO.parse('vhl-crispr-cas13-system.fasta', 'fasta'))
plasmid    = str(record.seq).upper()
vhl_seq    = plasmid[3029:].replace('T', 'U')  # same slice as Cell 2

print(f"VHL region length: {len(vhl_seq)} nt")

model.eval()
vhl_scores = []

with torch.no_grad():
    for i in range(10, len(vhl_seq) - 40, 1):
        window = vhl_seq[i-10 : i+40]
        if len(window) != 50:
            continue
        x     = torch.tensor(encode(window)).unsqueeze(0)
        score = model(x).item()
        vhl_scores.append((i, window[10:40], score))

vhl_scores.sort(key=lambda x: x[2], reverse=True)

print(f"Scored {len(vhl_scores)} target sites\n")
print(f"{'Position':<12} {'Score':<10} {'Sequence'}")
print("-" * 60)
for pos, seq, score in vhl_scores[:10]:
    print(f"{pos:<12} {score:.4f}     {seq}")

VHL region length: 543 nt
Scored 493 target sites

Position     Score      Sequence
------------------------------------------------------------
163          0.5341     ACUUUGACGGCGAGCCCCCGCCCUACCCCA
164          0.5341     CUUUGACGGCGAGCCCCCGCCCUACCCCAU
169          0.5341     ACGGCGAGCCCCCGCCCUACCCCAUUCUCC
170          0.5327     CGGCGAGCCCCCGCCCUACCCCAUUCUCCC
173          0.5309     CGAGCCCCCGCCCUACCCCAUUCUCCCUCC
187          0.5309     ACCCCAUUCUCCCUCCCGGUACAGGAAGGC
172          0.5309     GCGAGCCCCCGCCCUACCCCAUUCUCCCUC
174          0.5309     GAGCCCCCGCCCUACCCCAUUCUCCCUCCC
162          0.5308     AACUUUGACGGCGAGCCCCCGCCCUACCCC
171          0.5308     GGCGAGCCCCCGCCCUACCCCAUUCUCCCU


In [21]:
#--------------Intelligence Simulation using Cross-Attention Mechanisms and Stochiastic Stress Layers----------------#
#In order to prove the bindign I will build a stochastic stress layer.#
#The Stochastic Noise Layer will tajke into midn thermal Fluctuation, Molecular crowding ("viscosity" / association rate), and will also take into mind the pH shifts#
#Afetr all the data is colelcted I will run a MONTE CARLO Simulation#

In [48]:
# Step 1. Assign the Biophysical / Tensor Constants along with Sequence Pre-Processing#

GAS_CONSTANT_R = 8.314e-3 #kilojoules / (mol * K)
OPTIMAL_PH = 7.4  #In power of hydrogen#
BASAL_TEMP_K = 310.0 #Average body temperature in kelvin#
NUCLEOTIDE_MAP = {'g': 0, 't': 1, 'u': 1, 'a': 2, 'c': 3, 'n': 4}

def parse_sequence_to_tensor (fasta_string):
    """Converts a raw string sequence into an integer tensor for PyTorch."""
    # Clean sequence: skip header if exists, remove newlines, lower case
    lines = fasta_string.strip().split('\n')
    seq = "".join([line for line in lines if not line.startswith('>')]).lower()
    seq_ids = [NUCLEOTIDE_MAP.get(char, 4) for char in seq]
    return torch.tensor(seq_ids).unsqueeze(0)  # Adds batch dimension: [1, seq_len]



In [49]:
# Step 2. Creating the CasRx Intelligence Model#
class CasRxIntelligenceSim(nn.Module):
    def __init__(self, embed_dim = 64):
        super().__init__()

        self.embed_dim = embed_dim

        #The below creates 4 bases along with 1 unkown padding#

        self.nucleotide_embedding = nn.Embedding(5, embed_dim)

        #I will now create corss attention Weights for gRNA scanning the mRNA

        self.query_proj = nn.Linear(embed_dim, embed_dim)
        self.key_proj = nn.Linear(embed_dim, embed_dim)

        #I will now create a base affinity weight matrix to represent the CasRx Structural Grip#
        self.structural_grip = nn.Parameter(torch.ones(1))
    
    def compute_ph_penalty(self, current_ph):
        """This will calculate HEPN (Higher Eukaryotes and Prokaryotes Nucleotide-binding) charge state. If pH fluctuates from 7.4 the CasRx's Grip decreases."""
        penalty = torch.exp(-torch.tensor((current_ph - OPTIMAL_PH)**2))
        #This uses a Gaussian Penalty#
        return penalty
    
    def forward(self, target_mrna, grna_spacer, temp_k = 310.0, ph = 7.4, crowding = 1.0, noise_level = 0.0):
        """I will now run the inetraction simualtion under Biophysical Stress"""
        mrna_emb = self.nucleotide_embedding(target_mrna) 
        grna_emb = self.nucleotide_embedding(grna_spacer) 
        #The above embeds sequences into a vector space# 

        #I will now apply the stochastic noise I wrote earlier by creating random thermal kicks / cellular interference#
        if noise_level > 0:
            mrna_emb = mrna_emb + (torch.randn_like(mrna_emb) * noise_level)
            grna_emb = grna_emb + (torch.randn_like(grna_emb) * noise_level)

        Q = self.query_proj(grna_emb)
        K = self.key_proj(mrna_emb)

        # Raw binding enthalpy (dot product of sequence vectors)
        # Affinity of each gRNA nucleotide to each mRNA nucleotide
        H_raw = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.embed_dim)

        #I will now apply the biophysical stressors 
        grip_strength = self.structural_grip * self.compute_ph_penalty(ph)
        H_Adjusted = H_raw * grip_strength #This checks for pH which is the base structural grip of the protein#

        kinetic_energy = H_Adjusted / crowding #This checks for the molecular collisions / visvcosity since it reduces effective collisions / binding rate#

        beta = 1.0/ (GAS_CONSTANT_R * temp_k)
        binding_logits = kinetic_energy * beta #Thsi calculates the thermal fluctuation thorugh boltzmann distribution#

        site_affinities = binding_logits.sum(dim=1)
        grab_probabilities = torch.sigmoid(site_affinities) 
        #The above pools the attention across teh gRNA length to find the point / sequence which has the maximum bindign for the mRNA#

        #I will return the highest probability foudn on teh target mRNA#
        max_grab_prob, best_site_idx = torch.max(grab_probabilities, dim=-1)
        
        return max_grab_prob, best_site_idx

In [ ]:
# Step 3: Running a Monte Carlo Simulation#

def run_stress_test_proof():
    #usually the target_mRNA variable would be a real pateints mRNA Sequence#
    #However, for thsi simulation I will add a highly matching sequence in order to stress test#
   # 1. Load the Actual FASTA Data
    vhl_target_fasta = """>VHL CRISPR-CAS13 System
gtgaacggatcggcactgcgtgcgccaattctgcagacaaatggcagtattcatccacaattttaaaagaaaagggggg
attggggggtacagtgcaggggaaagaatagtagacataatagcaacagacatacaaactaaagaattacaaaaacaaa
ttacaaaaattcaaaattttcgggtttattacagggacagcagagatccagtttggttaattaatgcaaagatggataa
agttttaaacagagaggaatctttgcagctaatggaccttctaggtcttgaaaggagtgggaattggctccggtgcccg
tcagtgggcagagcgcacatcgcccacagtccccgagaagttggggggaggggtcggcaattgaaccggtgcctagaga
aggtggcgcggggtaaactgggaaagtgatgtcgtgtactggctccgcctttttcccgagggtgggggagaaccgtata
taagtgcagtagtcgccgtgaacgttctttttcgcaacgggtttgccgccagaacacaggtaagtgccgtgtgtggttc
ccgcgggcctggcctctttacgggttatggcccttgcgtgccttgaattacttccacctggctgcagtacgtgattctt
gatcccgagcttcgggttggaagtgggtgggagagttcgaggccttgcgcttaaggagccccttcgcctcgtgcttgag
ttgaggcctggcctgggcgctggggccgccgcgtgcgaatctggtggcaccttcgcgcctgtctcgctgctttcgataa
gtctctagccatttaaaatttttgatgacctgctgcgacgctttttttctggcaagatagtcttgtaaatgcgggccaa
gatctgcacactggtatttcggtttttggggccgcgggcggcgacggggcccgtgcgtcccagcgcacatgttcggcga
ggcggggcctgcgagcgcggccaccgagaatcggacgggggtagtctcaagctggccggcctgctctggtgcctggcct
cgcgccgccgtgtatcgccccgccctgggcggcaaggctggcccggtcggcaccagttgcgtgagcggaaagatggccg
cttcccggccctgctgcagggagctcaaaatggaggacgcggcgctcgggagagcgggcgggtgagtcacccacacaaa
ggaaaagggcctttccgtcctcagccgtcgcttcatgtgactccacggagtaccgggcgccgtccaggcacctcgatta
gttctcgagcttttggagtacgtcgtctttaggttggggggaggggttttatgcgatggagtttccccacactgagtgg
gtggagactgaagttaggccagcttggcacttgatgtaattctccttggaatttgccctttttgagtttggatcttggt
tcattctcaagcctcagacagtggttcaaagtttttttcttccatttcaggtgtcgtgacgtacggccaccatgagccc
caagaagaagagaaaggtggaggccagcatcgaaaaaaaaaagtccttcgccaagggcatgggcgtgaagtccacactc
gtgtccggctccaaagtgtacatgacaaccttcgccgaaggcagcgacgccaggctggaaaagatcgtggagggcgaca
gcatcaggagcgtgaatgagggcgaggccttcagcgctgaaatggccgataaaaacgccggctataagatcggcaacgc
caaattcagccatcctaagggctacgccgtggtggctaacaaccctctgtatacaggacccgtccagcaggatatgctc
ggcctgaaggaaactctggaaaagaggtacttcggcgagagcgctgatggcaatgacaatatttgtatccaggtgatcc
ataacatcctggacattgaaaaaatcctcgccgaatacattaccaacgccgcctacgccgtcaacaatatctccggcct
ggataaggacattattggattcggcaagttctccacagtgtatacctacgacgaattcaaagaccccgagcaccatagg
gccgctttcaacaataacgataagctcatcaacgccatcaaggcccagtatgacgagttcgacaacttcctcgataacc
ccagactcggctatttcggccaggcctttttcagcaaggagggcagaaattacatcatcaattacggcaacgaatgcta
tgacattctggccctcctgagcggactgaggcactgggtggtccataacaacgaagaagagtccaggatctccaggacc
tggctctacaacctcgataagaacctcgacaacgaatacatctccaccctcaactacctctacgacaggatcaccaatg
agctgaccaactccttctccaagaactccgccgccaacgtgaactatattgccgaaactctgggaatcaaccctgccga
attcgccgaacaatatttcagattcagcattatgaaagagcagaaaaacctcggattcaatatcaccaagctcagggaa
gtgatgctggacaggaaggatatgtccgagatcaggaaaaatcataaggtgttcgactccatcaggaccaaggtctaca
ccatgatggactttgtgatttataggtattacatcgaagaggatgccaaggtggctgccgccaataagtccctccccga
taatgagaagtccctgagcgagaaggatatctttgtgattaacctgaggggctccttcaacgacgaccagaaggatgcc
ctctactacgatgaagctaatagaatttggagaaagctcgaaaatatcatgcacaacatcaaggaatttaggggaaaca
agacaagagagtataagaagaaggacgccccgtcaccctggatgtgtcctgcctcaagggcgtcaccctggatgtgtcc
tgcctcaaggggcctcagttccccgtctgcaaaatggACCCCGGGGCGGTAGAGGGGCTTCAGACCGTGCTATCGTCCC
TGTACTAACTTTTTTTTTTTTTTTCAGATGCCAAGAAAAGCCGCATCCCCAGAGGAAGCCGGCGCCGAAGGCCCTCCTG
AGGAAGAGAAAGAAGCTGGGCGTCCCAGGCCAGTCCTAAGGAGCGTGAACTCACGGGAACCAGGGCAGGTTATATGCAA
TAGATCGCCCAGGGTCGTGTTGCCTTGGCTGAACTTTGACGGCGAGCCCCCGCCCTACCCCATTCTCCCTCCCGGTACA
GGAAGGCGGATTCATGCGTACCGAGGCCATCTCTGGCTTTTTCGGGACGCTGGAACCCACGACGGATTACTTCTCAATC
AGACTGAGCTGTTCGTTCCTTCACTGAACGTGGATGGGCAACCAATCTTCGCAAATATCACCCTGCCAGTAGTGTATAC
GCTAAAGGAACGCTGTCTGCTGGTGGTGCGAAGCAGTGTAAAGCCGGAGAACTATAGACGGCTGGACATCGTCCGCTCC
CTCTACGAAGACCTTGAGGATTACCCTAGTGTGCGCAAGGATATCCAAAGATTGAGCCAGGAGCACCTGGAATCTCAGC
ACCTGGAGGAGGAGCCC"""

    
    target_mrna_tensor = parse_sequence_to_tensor(vhl_target_fasta)
    
    # Target string found natively within the VHL construct
    grna_spacer_tensor = parse_sequence_to_tensor("ggcagtattcatccacaatt")

    #I will now initialize the model#
    model = CasRxIntelligenceSim(embed_dim=64)

    #Below are teh defined Monte Carlo Stress Conditions#
    conditions = [
        {"name": "Ideal Physiological", "temp": 310.0, "ph": 7.4, "crowd": 1.0, "noise": 0.05},
        {"name": "Mild Fever & Stress", "temp": 312.0, "ph": 7.2, "crowd": 1.2, "noise": 0.10},
        {"name": "Severe Acidosis & Fever", "temp": 314.0, "ph": 6.8, "crowd": 1.5, "noise": 0.25},
    ]

    monte_carlo_runs = 10000 #This is the number of stochastic permutations per environemnt condition#
    #The monte carlo will run 10000 itterations3

    print(f"Target mRNA Length: {target_mrna_tensor.size(1)} nt")
    print(f"gRNA Spacer Length: {grna_spacer_tensor.size(1)} nt\n")

    #I wil disabel gradient trackinf dor rapid simulation interference#

    with torch.no_grad():
        for cond in conditions:
            successes = 0
            culminative_prob = 0.0

            for _ in range(monte_carlo_runs):
                prob, idx = model(
                    target_mrna_tensor,
                    grna_spacer_tensor,
                    temp_k = cond["temp"],
                    ph = cond["ph"],
                    crowding = cond ["crowd"],
                    noise_level = cond["noise"]
                )

                culminative_prob += prob.item()

                #I will now defien a succsful grab as above 80% bindign probability#

                if prob.item() > 0.80:
                    successes += 1
            
            avg_prob = culminative_prob / monte_carlo_runs
            success_rate = (successes / monte_carlo_runs) * 100

            #------I will now do step 4 within teh step 3 cell because the reults are presented within the fro loop and with statment#
            #I will now print lal the values calculated through the monte carlo#
            #The 
            print(f"[{cond['name']}] -> Temp: {cond['temp']}K | pH: {cond['ph']} | Viscosity: {cond['crowd']}")
            print(f"  Average Grab Probability: {avg_prob:.4f}")
            print(f"  Monte Carlo Success Rate: {success_rate:.1f}%")
   
    if success_rate > 90.0:
        print("\nRESULT: PROVEN. The CasRx tensors successfully overcame the kinetic energy of the stress environment.")
    elif success_rate > 75.0:
        print("\nRESULT: MODERATELY PROVEN. The CasRx complex maintained functionality but showed vulnerability to extreme acidosis.")
    else:
        print("\nRESULT: FAILED. The CasRx complex dissociated under severe physiological stress (SNR dropped below threshold).")

In [51]:
#This prints otu all the values#
if __name__ == "__main__":
    run_stress_test_proof()

Target mRNA Length: 3572 nt
gRNA Spacer Length: 20 nt

[Ideal Physiological] -> Temp: 310.0K | pH: 7.4 | Viscosity: 1.0
  Average Grab Probability: 0.9556
  Monte Carlo Success Rate: 100.0%
[Mild Fever & Stress] -> Temp: 312.0K | pH: 7.2 | Viscosity: 1.2
  Average Grab Probability: 0.9329
  Monte Carlo Success Rate: 100.0%
[Severe Acidosis & Fever] -> Temp: 314.0K | pH: 6.8 | Viscosity: 1.5
  Average Grab Probability: 0.8640
  Monte Carlo Success Rate: 100.0%

RESULT: PROVEN. The CasRx tensors successfully overcame the kinetic energy of the stress environment.
